# 🛒 Phân Tích Dữ Liệu Lớn Thương Mại Điện Tử, Phân Cụm Khách Hàng RFM & Thực Nghiệm Benchmark SQL (IT4931 - Nhóm 3)

**Học phần:** IT4931 - Lưu trữ và Xử lý Dữ liệu Lớn  
**Giảng viên hướng dẫn:** TS. Vũ Tuyết Trinh  
**Nhóm thực hiện:** Nhóm 3 (Phạm Văn Quyết, Vũ Việt Tiến, Nguyễn Xuân Hồng Quân, Đặng Minh Quân)

Notebook này trình bày câu chuyện dữ liệu xuyên suốt từ bước kiểm định chất lượng dữ liệu thô (*Veracity*), tối ưu hóa lưu trữ hướng cột **Parquet/Snappy** và phân vùng (*Partitioning*), phân tích doanh thu theo lô (*Batch Layer*), phân cụm khách hàng **RFM K-Means**, xử lý luồng thời gian thực với **Sliding Window & Watermarking**, cho đến thực nghiệm đối chứng **Benchmark SQL** trên 3 mô hình lưu trữ.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.serving.db_client import ServingDatabaseClient

db = ServingDatabaseClient()
print('Connected Serving Layer Backend:', db.backend_name)

### Phân tích Kết quả Kết nối Tầng Serving
Hệ thống sử dụng kiến trúc **Polyglot Persistence** trong mô hình **Lambda Architecture**: dữ liệu lịch sử và nhật ký luồng được lưu trữ bất biến trên **HDFS/Parquet**, trong khi các khung nhìn tổng hợp (*Batch Views* và *Real-time Speed Views*) được đồng bộ vào cơ sở dữ liệu quan hệ tại tầng Serving để phục vụ truy vấn phân tích với độ trễ mili-giây.

In [ ]:
quality_df = db.read_table('data_quality_report')
storage_df = db.read_table('storage_comparison')
display(quality_df)
display(storage_df)

### Phân tích Chất lượng Dữ liệu (Veracity) & Hiệu quả Nén Lưu trữ Parquet/Snappy
- **Làm sạch dữ liệu (Data Cleansing)**: Tập dữ liệu thô (>1,000,000 bản ghi) chứa khoảng `1.2%` bản ghi lỗi bao gồm khuyết thiếu danh mục (`Category = NULL`), giá trị âm/bằng 0 (`Price <= 0`), và trùng khóa giao dịch (`Order_ID`). Quá trình chuyển đổi từ `Raw Data Layer` sang `Refined Data Layer` đã điền khuyết danh mục và loại bỏ hoàn toàn các giao dịch sai lệch trước khi ghi xuống tệp Parquet.
- **Hiệu quả nén hướng cột**: Khi chuyển từ định dạng văn bản `CSV thô (Baseline)` sang `Apache Parquet` kết hợp thuật toán nén khối `Snappy` và `Dictionary Encoding`, dung lượng lưu trữ giảm mạnh (tỷ lệ nén đạt ~`3.5x - 4.5x`) nhờ các cột lặp lại nhiều như `Category`, `Order_Status`, `Product_ID` được mã hóa từ điển cực kỳ hiệu quả.

In [ ]:
rev_df = db.read_table('batch_revenue_summary')
top10_df = db.read_table('batch_top_products')
seg_df = db.read_table('batch_customer_segments')
print('--- Top 10 Best-Selling Products ---')
display(top10_df)
print('--- RFM Customer Segments (K-Means Clustering) ---')
display(seg_df)

### Phân tích Kết quả Xử lý Khối (Batch Layer) & Phân cụm Học máy RFM K-Means
- **Top 10 Sản phẩm Bán chạy**: Truy vấn liên kết bảng (`INNER JOIN`) giữa hơn 1 triệu giao dịch và bảng danh mục `products_catalog` cho thấy nhóm sản phẩm điện tử (`electronics.smartphone`, `computers.notebook`) dẫn đầu về tổng doanh thu thực thu và đóng góp biên lợi nhuận cao.
- **Phân khúc Khách hàng RFM (Recency - Frequency - Monetary)**: Mô hình phân cụm **K-Means (`k=4`)** trên các đặc trưng đã chuẩn hóa (`StandardScaler`) phân tách rõ tập khách hàng thành 4 nhóm hành vi: nhóm `Champions (VIP High-Spend)` tuy chiếm tỷ lệ nhỏ về số lượng nhưng đóng góp tỷ trọng doanh thu vượt trội, trong khi nhóm `At-Risk / Hibernating` có thời gian mua hàng gần nhất (`avg_recency_days`) cao, cần chiến dịch kích hoạt lại.

In [ ]:
stream_df = db.read_table('streaming_window_metrics')
display(stream_df.tail(10))

### Phân tích Luồng Thời gian thực (Speed Layer: Sliding Window 1m/10s & Watermarking 10m)
Kết quả trên bảng `streaming_window_metrics` minh họa cơ chế **Event-Time Processing** với cửa sổ trượt **1 phút, trượt mỗi 10 giây**:
- Các sự kiện đến muộn (*Late Data*) do độ trễ mạng nhưng nằm trong ngưỡng **Watermark 10 phút** (`late_events_accepted`) vẫn được cập nhật chính xác vào cửa sổ thời gian xảy ra sự kiện.
- Các sự kiện đến trễ quá ngưỡng 10 phút (`late_events_dropped`) bị loại bỏ tự động để Spark giải phóng trạng thái cửa sổ cũ (*State Store Cleanup*), bảo đảm hệ thống vận hành liên tục không bị rò rỉ bộ nhớ.

In [ ]:
bench_df = db.read_table('benchmark_results')
rg_df = db.read_table('rowgroup_experiments')
display(bench_df[['query_id', 'storage_architecture', 'execution_time_sec', 'bytes_read_mb', 'cpu_utilization_pct', 'ram_usage_mb', 'speedup_vs_csv', 'io_reduction_pct']])
display(rg_df)

### Phân tích Thực nghiệm Benchmark SQL & Tối ưu Kích thước Row Group
1. **Đối chứng 3 Mô hình Lưu trữ**:
   - **(1) CSV thô (Baseline)**: Phải đọc toàn bộ tệp văn bản từ đĩa và phân tích cú pháp từng dòng cho mọi truy vấn (`io_reduction_pct = 0%`).
   - **(2) Parquet không phân vùng (Unpartitioned Parquet)**: Nhờ *Column Projection* (chỉ đọc các cột cần thiết) và *Predicate Pushdown* (`PushedFilters` dựa trên thống kê `min/max` tại Row Group Footer), lượng dữ liệu cần đọc giảm trên `80%` và tốc độ thực thi tăng gấp nhiều lần.
   - **(3) Parquet tối ưu hóa phân vùng (`year=/month=/`)**: Kết hợp cả **Partition Pruning (`PartitionFilters`)** và **Predicate Pushdown**, giúp các truy vấn lọc theo thời gian (`Q1`, `Q4`) bỏ qua ngay các thư mục tháng không liên quan, giảm tới **>95% dung lượng đọc I/O (`Bytes Read`)** và đạt tốc độ nhanh nhất.
2. **Ảnh hưởng của Kích thước Row Group (`parquet.block.size`)**:
   - Cấu hình **Row Group = 128 MB** khớp `1:1` với **HDFS Block Size = 128 MB** giúp tối ưu hóa tính cục bộ dữ liệu (*Data Locality*), tránh việc một Row Group bị chia cắt qua nhiều HDFS Block trên mạng mà vẫn duy trì hiệu quả lọc *Predicate Pushdown*.

## 🎯 Kết Luận Tổng Hợp

Hệ thống **Đường ống Dữ liệu Thương mại Điện tử Thời gian thực** của Nhóm 3 đã giải quyết trọn vẹn các mục tiêu đề ra trong đề xuất môn học **IT4931**:
1. **Kiến trúc Lưu trữ 3 Tầng trên HDFS**: Đảm bảo tính bất biến (*WORM*) ở tầng Raw, tối ưu hóa dung lượng nén (~4x) ở tầng Refined với Parquet + Snappy, và tăng tốc truy vấn vượt trội ở tầng Partitioned (`year=/month=/` và `event_type=`).
2. **Kiến trúc Xử lý Kép (Lambda Architecture)**: Kết hợp hoàn chỉnh nhánh **Batch Processing** (Spark SQL + Catalyst/CBO + Spark MLlib RFM K-Means) trên hơn 1 triệu giao dịch lịch sử và nhánh **Speed Processing** (Kafka + Spark Structured Streaming với Sliding Window 1m/10s và Watermark 10m).
3. **Chứng minh Thực nghiệm Rõ ràng**: Bộ thực nghiệm Benchmark SQL đã định lượng chính xác lợi ích của **Predicate Pushdown** và **Partition Pruning** thông qua 4 chỉ số: *Execution Time*, *Scan Volume (Bytes Read)*, *CPU Utilization*, và *RAM Usage*, đồng thời hiển thị trực quan trên **Streamlit Dashboard**.